In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Load Data
print("Loading datasets...")
df1 = pd.read_csv("Flights_2022_5.csv", low_memory=False)
df2 = pd.read_csv("Flights_2022_6.csv", low_memory=False)
df3 = pd.read_csv("Flights_2022_7.csv", low_memory=False)
df4 = pd.read_csv("Flights_2022_8.csv", low_memory=False)

Loading datasets...


In [3]:
# 2. Combine Data
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
print(f"Initial Shape: {df.shape}")

Initial Shape: (2437446, 120)


In [4]:
# 3. Drop completely empty columns and unnecessary time columns
df.drop(['Year', 'Quarter'], axis=1, inplace=True, errors='ignore')
df.dropna(axis=1, how='all', inplace=True)


In [5]:
# 4. Remove Duplicate Rows
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Dropped {initial_rows - df.shape[0]} duplicate rows.")

Dropped 0 duplicate rows.


In [6]:
# 5. Handle Delay Columns (Fill NaNs with 0 for non-delayed flights)
delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
existing_delay_cols = [col for col in delay_cols if col in df.columns]
if existing_delay_cols:
    df[existing_delay_cols] = df[existing_delay_cols].fillna(0)

In [7]:
# 6. Handle Cancellation Codes
if 'CancellationCode' in df.columns:
    df['CancellationCode'] = df['CancellationCode'].fillna('N/A')

# 7. Filter out invalid/unusable rows
# If a flight wasn't cancelled or diverted, it must have departure and arrival times.
# This drops rows that have missing operational data due to logging errors.
if 'Cancelled' in df.columns and 'Diverted' in df.columns:
    critical_cols = ['DepTime', 'ArrTime', 'ActualElapsedTime']
    existing_crit = [col for col in critical_cols if col in df.columns]
    
    # Keep row if it was cancelled/diverted OR if it has valid flight times
    is_cancelled_or_diverted = (df['Cancelled'] == 1) | (df['Diverted'] == 1)
    has_valid_times = df[existing_crit].notnull().all(axis=1)
    
    df = df[is_cancelled_or_diverted | has_valid_times]

# 8. Optimize Data Types (Saves massive memory)
# Convert binary flags to boolean
bool_cols = ['Cancelled', 'Diverted']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

# Downcast floats and ints where possible
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')

# Convert low-cardinality strings (like Carrier, Origin, Dest) to categories
cat_cols = ['Marketing_Airline_Network', 'Operating_Airline', 'Origin', 'Dest', 'Tail_Number']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 9. Parse Dates (if applicable)
if 'FlightDate' in df.columns:
    df['FlightDate'] = pd.to_datetime(df['FlightDate'])

In [8]:
# 10. Final Summary
print("\n--- Cleaning Complete ---")
print(f"Final Shape: {df.shape}")
print("\nRemaining Missing Values per Column:")
print(df.isnull().sum())


--- Cleaning Complete ---
Final Shape: (2437446, 93)

Remaining Missing Values per Column:
Month                              0
DayofMonth                         0
DayOfWeek                          0
FlightDate                         0
Marketing_Airline_Network          0
                              ...   
Div2TotalGTime               2437411
Div2LongestGTime             2437411
Div2WheelsOff                2437433
Div2TailNum                  2437433
Duplicate                          0
Length: 93, dtype: int64


In [9]:
# Calculate the threshold (e.g., keep columns with at least 10% non-null values)
threshold = 0.10 * len(df)
df.dropna(axis=1, thresh=threshold, inplace=True)

print(f"Shape after dropping columns with >90% missing values: {df.shape}")

Shape after dropping columns with >90% missing values: (2437446, 66)


In [10]:
# # Print every column name in a clean list
print("\n--- Remaining Columns ---")
import pprint
pprint.pprint(df.columns.tolist(), compact=True)


--- Remaining Columns ---
['Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network',
 'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline',
 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline',
 'Operating_Airline ', 'DOT_ID_Operating_Airline',
 'IATA_Code_Operating_Airline', 'Tail_Number',
 'Flight_Number_Operating_Airline', 'OriginAirportID', 'OriginAirportSeqID',
 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState',
 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID',
 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState',
 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime',
 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime',
 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups',
 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime

In [11]:
print(df.shape)


(2437446, 66)


In [12]:
print(df.isnull().sum())


Month                        0
DayofMonth                   0
DayOfWeek                    0
FlightDate                   0
Marketing_Airline_Network    0
                            ..
NASDelay                     0
SecurityDelay                0
LateAircraftDelay            0
DivAirportLandings           0
Duplicate                    0
Length: 66, dtype: int64


In [13]:
# Force Pandas to show all columns side-by-side
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("\n--- Random 10-Row Sample of the Cleaned Data ---")
print(df.sample(15))


--- Random 10-Row Sample of the Cleaned Data ---
         Month  DayofMonth  DayOfWeek FlightDate Marketing_Airline_Network Operated_or_Branded_Code_Share_Partners  DOT_ID_Marketing_Airline IATA_Code_Marketing_Airline  Flight_Number_Marketing_Airline Operating_Airline   DOT_ID_Operating_Airline IATA_Code_Operating_Airline Tail_Number  Flight_Number_Operating_Airline  OriginAirportID  OriginAirportSeqID  OriginCityMarketID Origin      OriginCityName OriginState  OriginStateFips OriginStateName  OriginWac  DestAirportID  DestAirportSeqID  DestCityMarketID Dest           DestCityName DestState  DestStateFips DestStateName  DestWac  CRSDepTime  DepTime  DepDelay  DepDelayMinutes  DepDel15  DepartureDelayGroups DepTimeBlk  TaxiOut  WheelsOff  WheelsOn  TaxiIn  CRSArrTime  ArrTime  ArrDelay  ArrDelayMinutes  ArrDel15  ArrivalDelayGroups ArrTimeBlk  Cancelled CancellationCode  Diverted  CRSElapsedTime  ActualElapsedTime  AirTime  Flights  Distance  DistanceGroup  CarrierDelay  WeatherDelay  